In [19]:
!pip uninstall -y dgl torchdata

!pip install -q torch==2.2.0+cu121 --extra-index-url https://download.pytorch.org/whl/cu121

# Install torchdata compatible with torch 2.2
!pip install -q torchdata==0.7.1

# Install DGL for cu121
!pip install dgl -q -f https://data.dgl.ai/wheels/torch-2.2/cu121/repo.html
!pip install numpy
!pip install rdkit

Found existing installation: dgl 2.4.0+cu121
Uninstalling dgl-2.4.0+cu121:
  Successfully uninstalled dgl-2.4.0+cu121
Found existing installation: torchdata 0.7.1
Uninstalling torchdata-0.7.1:
  Successfully uninstalled torchdata-0.7.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchtune 0.6.1 requires torchdata==0.11.0, but you have torchdata 0.7.1 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.2/37.2 MB 52.8 MB/s eta 0:00:00:00:0100:01


In [20]:
%%writefile /kaggle/working/egnn_layer.py
import torch
import torch.nn as nn
import dgl
import dgl.function as fn


class EGNNLayer(nn.Module):
    """
    One layer of an Equivariant Graph Neural Network (EGNN).
    Based on: "E(n) Equivariant Graph Neural Networks" (Satorras et al., 2021)

    What makes EGNN different from GINEConv:
    - GINEConv: uses pre-computed distances as extra edge features. Coordinates
      are static — they never change during message passing.
    - EGNN: computes distances live from pos during every forward pass, AND
      updates the 3D coordinates of every node as part of the layer itself.
      This means the geometry evolves as information flows through the network,
      making it sensitive to the actual 3D shape of the molecule.

    Per-layer operations:
    1. For every edge: compute distance from current pos, run edge MLP
    2. For every node: aggregate neighbour messages, run node MLP → new hidden state
    3. For every node: compute a weighted sum of relative position vectors → update pos
    """

    def __init__(self, hidden_dim, edge_attr_dim=5):
        super().__init__()

        # Edge MLP: takes [h_i, h_j, distance, edge_attr] → message
        # hidden_dim * 2 for the two node states + 1 for distance + edge_attr_dim for bond type
        self.edge_mlp = nn.Sequential(
            nn.Linear(hidden_dim * 2 + 1 + edge_attr_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
        )

        # Node MLP: takes [h_i, aggregated messages] → new h_i
        self.node_mlp = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )

        # Coordinate MLP: takes edge message → scalar weight for pos update
        # Output is a single scalar that scales the relative position vector (pos_i - pos_j)
        self.coord_mlp = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, 1),
        )

    def edge_message(self, edges):
        """
        Runs on every edge simultaneously.
        Computes the distance between the two endpoint atoms from their
        current positions, then feeds everything into the edge MLP.
        """
        # Relative position vector and its scalar distance
        rel_pos  = edges.src['pos'] - edges.dst['pos']          # (E, 3)
        distance = torch.norm(rel_pos, dim=-1, keepdim=True)    # (E, 1)

        # Concatenate: source hidden state, dest hidden state, distance, bond type
        edge_input = torch.cat([
            edges.src['h'],           # (E, hidden_dim)
            edges.dst['h'],           # (E, hidden_dim)
            distance,                  # (E, 1)
            edges.data['edge_attr'],   # (E, edge_attr_dim)
        ], dim=-1)

        message    = self.edge_mlp(edge_input)     # (E, hidden_dim)
        coord_weight = self.coord_mlp(message)     # (E, 1) — scalar for pos update

        return {
            'message':      message,
            'coord_weight': coord_weight * rel_pos,  # (E, 3) — weighted relative vector
        }

    def node_update(self, nodes):
        """
        Runs on every node simultaneously.
        Aggregates incoming messages and updates the node's hidden state.
        """
        # 'agg_msg' is the sum of all incoming messages (set by dgl after edge_message)
        node_input = torch.cat([nodes.data['h'], nodes.data['agg_msg']], dim=-1)
        new_h      = self.node_mlp(node_input)
        return {'h': new_h}

    def forward(self, g, h, pos, edge_attr):
        with g.local_scope():
            g.ndata['h']         = h
            g.ndata['pos']       = pos
            g.edata['edge_attr'] = edge_attr

            # Step 1: compute messages and coordinate weights along every edge
            g.apply_edges(self.edge_message)

            # Step 2: aggregate messages into each node
            g.update_all(fn.copy_e('message', 'm'), fn.sum('m', 'agg_msg'))

            # Step 3: update node hidden states
            g.apply_nodes(self.node_update)

            # Step 4: update coordinates
            # Sum the weighted relative vectors arriving at each node
            g.update_all(fn.copy_e('coord_weight', 'cw'), fn.sum('cw', 'agg_cw'))
            new_pos = pos + g.ndata['agg_cw']   # (N, 3)

            return g.ndata['h'], new_pos


Overwriting /kaggle/working/egnn_layer.py


In [21]:
%%writefile /kaggle/working/data_loader.py
import torch
import dgl
from torch.utils.data import Dataset, DataLoader, random_split


# ===========================================================================
#  WRAPPER: Makes the list of (dgl.graph, label) tuples behave like a
#  proper PyTorch Dataset so random_split and DataLoader work on it.
# ===========================================================================
class MoleculeDataset(Dataset):
    def __init__(self, data):
        self.data = data  # list of (dgl.DGLGraph, y_tensor) tuples

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]


def collate_fn(batch):
    """
    dgl.batch() merges N separate molecule graphs into one large disconnected
    graph, tracking which nodes/edges belong to which molecule internally.
    This is the DGL equivalent of PyG's automatic Batch.from_data_list().
    """
    graphs, labels = zip(*batch)
    batched_graph  = dgl.batch(graphs)
    batched_labels = torch.stack(labels, dim=0).squeeze(1)
    return batched_graph, batched_labels


# ===========================================================================
#  TOX21 LOADERS  (classification — 12 binary labels)
# ===========================================================================
def get_tox21_loaders(path=r"/kaggle/input/datasets/prajwalnayakat/molecolyte-datasets/tox21_3d_egnn_dataset.pt", batch_size=32):
    raw     = torch.load(path, weights_only=False)
    dataset = MoleculeDataset(raw)

    total = len(dataset)
    train_size = int(0.8 * total)
    val_size   = int(0.1 * total)
    test_size  = total - train_size - val_size

    print(f"Tox21 Split → Train: {train_size} | Val: {val_size} | Test: {test_size}")

    train_ds, val_ds, test_ds = random_split(dataset, [train_size, val_size, test_size])

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  collate_fn=collate_fn)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

    return train_loader, val_loader, test_loader


# ===========================================================================
#  QM9 LOADERS  (regression — 19 quantum properties, we train on column 7)
# ===========================================================================
def get_qm9_loaders(path=r"/kaggle/input/datasets/prajwalnayakat/molecolyte-datasets/qm9_3d_egnn_dataset.pt", batch_size=32):
    raw     = torch.load(path, weights_only=False)
    dataset = MoleculeDataset(raw)

    total = len(dataset)
    train_size = int(0.8 * total)
    val_size   = int(0.1 * total)
    test_size  = total - train_size - val_size

    print(f"QM9 Split → Train: {train_size} | Val: {val_size} | Test: {test_size}")

    train_ds, val_ds, test_ds = random_split(dataset, [train_size, val_size, test_size])

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  collate_fn=collate_fn)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

    return train_loader, val_loader, test_loader


# ===========================================================================
#  EXPOSE LOADERS FOR IMPORT
#  train_QM9.py and train_Tox21.py import directly from here.
# ===========================================================================
tox21_train_loader, tox21_val_loader, tox21_test_loader = get_tox21_loaders()
qm9_train_loader,   qm9_val_loader,   qm9_test_loader   = get_qm9_loaders()


# ===========================================================================
#  VERIFICATION — run this file directly to confirm batches look right
# ===========================================================================
if __name__ == "__main__":
    print("\n--- Tox21 Batch Check ---")
    for batch_graph, batch_labels in tox21_train_loader:
        print(f"Graphs in batch      : {batch_graph.batch_size}")
        print(f"Total nodes          : {batch_graph.num_nodes()}")
        print(f"Total edges          : {batch_graph.num_edges()}")
        print(f"Node feature shape   : {batch_graph.ndata['x'].shape}")
        print(f"Position shape       : {batch_graph.ndata['pos'].shape}")
        print(f"Edge attr shape      : {batch_graph.edata['edge_attr'].shape}")
        print(f"Label shape          : {batch_labels.shape}")   # should be (32, 12)
        break

    print("\n--- QM9 Batch Check ---")
    for batch_graph, batch_labels in qm9_train_loader:
        print(f"Graphs in batch      : {batch_graph.batch_size}")
        print(f"Total nodes          : {batch_graph.num_nodes()}")
        print(f"Total edges          : {batch_graph.num_edges()}")
        print(f"Node feature shape   : {batch_graph.ndata['x'].shape}")
        print(f"Position shape       : {batch_graph.ndata['pos'].shape}")
        print(f"Edge attr shape      : {batch_graph.edata['edge_attr'].shape}")
        print(f"Label shape          : {batch_labels.shape}")   # should be (32, 19)
        break

Overwriting /kaggle/working/data_loader.py


In [22]:
import os
os.environ["DGL_SKIP_GRAPHBOLT"] = "1"

import torch
import time
import torch.nn as nn
import dgl
from sklearn.metrics import roc_auc_score
from data_loader import tox21_test_loader
from egnn_layer import EGNNLayer


# ==========================================
# MODEL ARCHITECTURE (must match train_Tox21.py exactly)
# ==========================================
class MoleColyteEGNN(nn.Module):
    def __init__(self, in_node_features=8, hidden_dim=128, edge_attr_dim=5, num_layers=3, out_features=12):
        super().__init__()

        self.input_proj = nn.Linear(in_node_features, hidden_dim)

        self.egnn_layers = nn.ModuleList([
            EGNNLayer(hidden_dim=hidden_dim, edge_attr_dim=edge_attr_dim)
            for _ in range(num_layers)
        ])

        self.prediction_head = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Linear(64, out_features)
        )

    def forward(self, g, x, pos, edge_attr):
        h = self.input_proj(x)

        for layer in self.egnn_layers:
            h, pos = layer(g, h, pos, edge_attr)

        g.ndata['h'] = h
        mol_embedding = dgl.mean_nodes(g, 'h')

        return self.prediction_head(mol_embedding)


def evaluate():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Kiln: {device}")

    model = MoleColyteEGNN(in_node_features=8, hidden_dim=128, edge_attr_dim=5, num_layers=3, out_features=12)
    model.load_state_dict(
        torch.load(
            r"/kaggle/input/models/prajwalnayakat/egnn-tox21/pytorch/default/1/molecolyte_egnn_tox21_best.pt",
            weights_only=False,
            map_location=device
        )
    )
    model = model.to(device)
    model.eval()

    all_preds   = []
    all_targets = []

    print("Running inference on the unseen Test Set...")
    with torch.no_grad():
        for batch_graph, batch_labels in tox21_test_loader:
            batch_graph  = batch_graph.to(device)
            batch_labels = batch_labels.to(device)

            x         = batch_graph.ndata['x'].to(torch.float)
            pos       = batch_graph.ndata['pos'].to(torch.float)
            edge_attr = batch_graph.edata['edge_attr'].to(torch.float)

            logits = model(batch_graph, x, pos, edge_attr)   # (batch, 12)
            probs  = torch.sigmoid(logits)

            all_preds.append(probs.cpu())
            all_targets.append(batch_labels.cpu())

    # Concatenate all batches — stay in pure torch
    all_preds   = torch.cat(all_preds,   dim=0)   # (N, 12)
    all_targets = torch.cat(all_targets, dim=0)   # (N, 12)

    roc_aucs = []

    print("\n=== MoleColyte EGNN — Tox21 Evaluation Results ===")
    for i in range(12):
        # NaN mask in pure torch
        valid_mask   = all_targets[:, i] == all_targets[:, i]
        task_targets = all_targets[valid_mask, i].tolist()   # plain Python list, zero numpy
        task_preds   = all_preds[valid_mask,   i].tolist()   # plain Python list, zero numpy

        if len(set(task_targets)) > 1:
            score = roc_auc_score(task_targets, task_preds)
            roc_aucs.append(score)
            print(f"Assay {i + 1:02d} AUC-ROC: {score:.4f}")
        else:
            print(f"Assay {i + 1:02d} Skipped (Only one class present in test set)")

    print("--------------------------------------------------")
    mean_auc = sum(roc_aucs) / len(roc_aucs)
    print(f"🏆 FINAL MEAN AUC-ROC: {mean_auc:.4f}")


if __name__ == "__main__":
    since = time.time()
    evaluate()
    print(f"Time: {(time.time() - since) / 60:.2f} minutes.")

Kiln: cpu
Running inference on the unseen Test Set...

=== MoleColyte EGNN — Tox21 Evaluation Results ===
Assay 01 AUC-ROC: 0.7382
Assay 02 AUC-ROC: 0.7938
Assay 03 AUC-ROC: 0.8458
Assay 04 AUC-ROC: 0.6949
Assay 05 AUC-ROC: 0.6300
Assay 06 AUC-ROC: 0.6802
Assay 07 AUC-ROC: 0.8215
Assay 08 AUC-ROC: 0.7714
Assay 09 AUC-ROC: 0.7926
Assay 10 AUC-ROC: 0.7546
Assay 11 AUC-ROC: 0.8359
Assay 12 AUC-ROC: 0.8207
--------------------------------------------------
🏆 FINAL MEAN AUC-ROC: 0.7650
Time: 0.01 minutes.
